In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

In [ ]:
import numpy as np  # importa o NumPy para operações numéricas e trigonométricas

def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1.5, 
    step_size=1,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


### Tratamento de fundo + Otsu

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    # piorou apenas com otsu sem média global
    # 1) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)

    # 2) Binarização Otsu
    _, otsu = cv2.threshold(
        no_bg_global, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    # 3) Retornar vetor flatten binário
    vector = (otsu.flatten() > 0).astype(np.uint8)

    return vector


In [ ]:
import cv2
import os
import glob
import numpy as np
import sys
import matplotlib.pyplot as plt
from wisardpkg import ClusWisard

# Caminhos
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

# ============================================================
# Carrega imagens
# ============================================================
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truth
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# ============================================================
# Geração de regiões circulares
# ============================================================
def generate_search_regions_circular(
    prev_bbox,
    frame_shape,
    search_region_scale=1.5,
    step_size=5,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior.
    step_size define deslocamento real em pixels.
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2
    raio_max = (max(w, h) / 2) * search_region_scale

    # Retorna primeiro o bbox original
    yield (x, y, w, h)

    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))

        direction = -1 if clockwise else 1
        thetas = start_angle + direction * np.linspace(
            0, 2 * np.pi, num_steps, endpoint=False
        )

        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)

# ============================================================
# MAIN — primeiro frame
# ============================================================
# Lê o primeiro frame
first_frame_path = image_paths[200]
first_frame = cv2.imread(first_frame_path)

print("Primeiro frame:", first_frame_path)
print("Shape do frame:", first_frame.shape)

# BBox inicial (x, y, w, h)
prev_bbox = tuple(map(int, all_ground_truths[200]))
print("BBox inicial:", prev_bbox)

# Gera regiões de busca
search_regions = list(
    generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=first_frame.shape,
        search_region_scale=1.5,
        step_size=5,
        start_angle=0,
        clockwise=True
    )
)

# Printar regiões
print(f"\nTotal de regiões geradas: {len(search_regions)}")
for i, bbox in enumerate(search_regions[:20]):
    print(f"Região {i}: {bbox}")

# ============================================================
# Visualização
# ============================================================
frame_vis = first_frame.copy()

for (x, y, w, h) in search_regions:
    cv2.rectangle(frame_vis, (x, y), (x + w, y + h), (0, 255, 0), 1)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB))
plt.title("Regiões de busca circulares - Primeiro Frame")
plt.axis("off")
plt.show()


In [ ]:
import math

patch_regions = []   # <-- lista para guardar os patches
patch_indices = []   # opcional: guarda o índice da região

max_regions = 2000
cols = 10
rows = math.ceil(max_regions / cols)

plt.figure(figsize=(cols * 3, rows * 3))

count = 0
for i, region in enumerate(search_regions):
    patch_region = tracker_utils.extract_patch(first_frame, region)
    if patch_region.size == 0:
        continue

    patch_regions.append(patch_region)
    patch_indices.append(i)

    count += 1
    ax = plt.subplot(rows, cols, count)
    ax.imshow(cv2.cvtColor(patch_region, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Região {i}", fontsize=7)
    ax.axis("off")

    if count >= max_regions:
        break

plt.suptitle("Patches das regiões de busca", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Total de patches válidos armazenados: {len(patch_regions)}")


In [ ]:
import cv2
import numpy as np

def preprocess_frame(patch, min_area_ratio=0.30):
    # 1) Grayscale
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    else:
        gray = patch.copy()

    h, w = gray.shape
    total_area = h * w
    min_area = min_area_ratio * total_area

    # 2) Binarização
    _, binary = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 3) Componentes conectados
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    # 4) Máscara final
    mask = np.zeros_like(gray, dtype=np.uint8)

    for i in range(1, num_labels):  # ignora fundo
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask[labels == i] = 1  # já binário

    # 5) Vetor binarizado (flatten)
    vector = mask.flatten().astype(np.uint8)

    return vector


In [ ]:
for i, region in enumerate(search_regions):
    patch_region = tracker_utils.extract_patch(first_frame, region)
    if patch_region.size == 0:
        continue

    fg = preprocess_frame(patch_region)



In [ ]:
# Recupera altura e largura original do patch
h, w = patch_regions[0].shape[:2]

max_show = 137
cols = 10
rows = math.ceil(max_show / cols)

plt.figure(figsize=(cols * 2, rows * 2))

for i in range(min(max_show, len(preprocessed_vectors))):
    binary_img = preprocessed_vectors[i].reshape(h, w)

    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(binary_img, cmap="gray")
    ax.set_title(f"Região {patch_indices[i]}", fontsize=7)
    ax.axis("off")

plt.suptitle("Patches após preprocess_frame (binarizados)", fontsize=14)
plt.tight_layout()
plt.show()
